In [1]:
import scanpy
import treeclust.scanpy as tcsc

from treeclust.pipelines import PipelineBootstrapper
from treeclust.decomposition import PCA
from sklearn.model_selection import ShuffleSplit

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

/Users/gatocor/miniforge3/envs/treeclust-scanpy/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
folder = "data"
# file = "48h"
# file = "72h"
# file = "96h"
# file = "120h"
# file = "144h"
file = "full"

resolutions_max = {
    "48h": 1,
    "72h": 1,
    "96h": 1,
    "120h": 1.5,
    "144h": 1.5,
    "full": 1.5
}

In [8]:
if file != "full":

    file_path = f"{folder}/dias_{file}_EM_processed.h5ad"

    adata = scanpy.read_h5ad(file_path)

    bootstrapper = PipelineBootstrapper(
            steps=[
                ("pca", PCA(n_components=30))
            ],
            observation_splitter=ShuffleSplit(test_size=0.2),
            feature_splitter=ShuffleSplit(test_size=0.2)
        )

    tcsc.pp.consensus_neighbors(
            adata,
            n_neighbors=15,
            use_rep="X",
            key_added="neighbors_consensus",
            n_splits=20,
            use_highly_variable=True,
            pipeline_bootstrapper=bootstrapper,
        )

    tcsc.tl.multiresolution_leiden(
            adata,
            resolutions=(0,resolutions_max[file]),
            neighbors_key="neighbors_consensus",
            key_added="leiden_tc",
            random_state=0,
        )
    
else:

    file_path = f"{folder}/dias_{file}_processed.h5ad"

    adata = scanpy.read_h5ad(file_path)

    bootstrapper = PipelineBootstrapper(
            steps=[],
            observation_splitter=ShuffleSplit(test_size=0.2),
            # feature_splitter=ShuffleSplit(test_size=0.2)
        )

    tcsc.pp.consensus_neighbors(
            adata,
            n_neighbors=15,
            use_rep="X_scvi",
            key_added="neighbors_consensus",
            n_splits=20,
            use_highly_variable=True,
            pipeline_bootstrapper=bootstrapper,
        )

    tcsc.tl.multiresolution_leiden(
            adata,
            resolutions=(0,resolutions_max[file]),
            neighbors_key="neighbors_consensus",
            key_added="leiden_tc",
            random_state=0,
        )

/Users/gatocor/miniforge3/envs/treeclust-scanpy/lib/python3.13/site-packages/anndata/_core/anndata.py:1796: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
CNN Bootstrap (n_neighbors=15): 100%|██████████| 20/20 [00:30<00:00,  1.52s/it]


Generating resolution profile from 0 to 1.5...


228it [1:12:40, 19.12s/it, resolution_parameter=0.0234]


Found 16 transition boundaries
Generated 18 stable resolution values


283it [2:50:42, 36.19s/it, resolution_parameter=0.131]
Multiresolution Leiden: 100%|██████████| 18/18 [17:27<00:00, 58.22s/resolution]


In [9]:
adata.uns["leiden_tc"]["resolution_range"] = list(adata.uns["leiden_tc"]["resolution_range"])
adata.uns["leiden_tc_params"]["resolution_range"] = list(adata.uns["leiden_tc_params"]["resolution_range"])

In [10]:
adata.write(f"{folder}/dias_{file}_EM_processed_annotated.h5ad")